# Motivation and introduction

This file provides examples on how to use the commandline interface, insofar 
as the help function and [cmd help](../000_specs/002_cmd.md) do not suffice.

In this example file, we call the cli via the cli.run function.  
All command are also available from outside python via the shell. 
a call to  
'cli.run(["arg1", "arg2"])' within python  
is equivalent to  
'uv run pylib arg1 arg2' in the shell  

In [1]:
# imports and preparations
from pylib import cli as libcli

# lets enable autoreload so we can use these scrips for interactive src-debugging.
%load_ext autoreload
%autoreload 2

cli = libcli.getCLI()
Tp,tp,ta,to,ant = cli.getTyperShortcuts()

# adding functions to the cli

In [2]:
#by default, the cli provides all functions it found in the src/<packagename>/fns folder.
#it finds any functions decorated with the cli.cmd decorator.

libcli.run(["--help"]) #throws a systeExit, which is ok (#TODO: maybe fix for cosmetics sake?)

                                                                                                                   
 Usage: api [OPTIONS] COMMAND [ARGS]...                                                                            
                                                                                                                   

╭─ Options ───────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ --version             -v            Show the version of this script                                             │
│ --docu                              Show docu                                                                   │
│ --flag                        TEXT  Flag(s) that are available in code via lib.getFlags(). See --flag --help    │
│                                     for list of availables. Pass multiple times for multiple flags.             │
│ --install-completion                Install completion for the current shell.                                   │
│ --show-completion                   Show completion for the current shell, to copy it or customize the          │
│                                     installation.                                                               │
│ --help                -h            Show this message and exit.                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Commands ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ history          Recall cli history (mainly used for debugging).                                                │
│ create-package   Create a package that contains all the boilerplate provided by pylib.                          │
│ extract-lib      Extract the pylib code from the given path.                                                    │
│ hello-world      Print hello world. Minimal example of a command.                                               │
│ inject-lib       Inject the pylib code into the given path.                                                     │
│ tui              Display the commandline interface inside a tui.                                                │
│ dev              contains development subcommands                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SystemExit: 0

C:\data\userdata\004_code\pylib\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
#lets define a new cmdline function and add it
#use arguments and options as is usual in typer. see lib/fns for examples.

@cli.cmd
def example_cmd(echotext: ant[str, ta(help="string that will be echoed")] = "default"):
    """An example cmd."""
    print(echotext)

In [13]:
#now that we defined the new cmd, call the help again. It will show up there
libcli.run(["--help"])

                                                                                                                   
 Usage: api [OPTIONS] COMMAND [ARGS]...                                                                            
                                                                                                                   

╭─ Options ───────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ --version             -v            Show the version of this script                                             │
│ --docu                              Show docu                                                                   │
│ --flag                        TEXT  Flag(s) that are available in code via lib.getFlags(). See --flag --help    │
│                                     for list of availables. Pass multiple times for multiple flags.             │
│ --install-completion                Install completion for the current shell.                                   │
│ --show-completion                   Show completion for the current shell, to copy it or customize the          │
│                                     installation.                                                               │
│ --help                -h            Show this message and exit.                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Commands ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ history          Recall cli history (mainly used for debugging).                                                │
│ create-package   Create a package that contains all the boilerplate provided by pylib.                          │
│ extract-lib      Extract the pylib code from the given path.                                                    │
│ hello-world      Print hello world. Minimal example of a command.                                               │
│ inject-lib       Inject the pylib code into the given path.                                                     │
│ tui              Display the commandline interface inside a tui.                                                │
│ example-cmd      An example cmd.                                                                                │
│ dev              contains development subcommands                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SystemExit: 0

In [12]:
# or call the help directly on the command
libcli.run(["example-cmd","-h"])

[]


                                                                                                                   
 Usage: api example-cmd [OPTIONS] [ECHOTEXT]                                                                       
                                                                                                                   

An example cmd.

╭─ Arguments ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│   echotext      [ECHOTEXT]  string that will be echoed [default: default]                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Options ───────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ --help  -h        Show this message and exit.                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SystemExit: 0

In [4]:
# execute the new command:
libcli.run(["example-cmd"])

[]
default


SystemExit: 0

__synopsis, further reading__

Using the cli.cmd decorator, you can generate a commandline function out of every callable you define anywhere.  
The documentation will be generated from the function paramters itself, as is customary in typer.  
These functions will also automatically be added in the TUI, and by extension the web-TUI.  
#TODO: add notes on autoimport from folders

# adding flags to the cli

Flags are global modifiers for behavior of the code. They are especially useful for debugging, or seldom used edgecases.   
Once a flag is defined, it can be appended to a commandline call and can be queried in code.  
Note that in the commandline, flags must be passed before subcommands. Otherwise, they are passed as options to the subcommand.

In [5]:
#lets define flags
cli.addFlag("testflag", help="a flag for demonstration", type=bool, default=False)
cli.addFlag("addText", help= "additional text to be printed", type=str, default = "")

#once defined, the help can display all registered flags
libcli.run(["--flag", "--help"])

Registered commandline flags:

-- addtext / additional text to be printed / str

-- testflag / a flag for demonstration / bool

In [14]:
#lets define a function that changes its behavior dependent on flags
@cli.cmd
def example_cmd_with_flag(echotext: ant[str, ta(help="string that will be echoed")] = "default"):
    """An example cmd."""
    f = cli.getFlag("testflag")
    if f:
        print(echotext[::-1])
    else:
        print(echotext)
    at = cli.getFlag("addText")
    if at:
        print(at)

    # see examples below for details on this
    allflags = cli.getFlag()
    unknowns = allflags.get("_")
    if unknowns:
        print(unknowns)

In [4]:
#calling the fn without the flag
libcli.run(["example-cmd-with-flag", "flagecho"])

flagecho


SystemExit: 0

C:\data\userdata\004_code\pylib\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [15]:
#calling the fn with the flag
libcli.run([ "--flag", "testflag", "example-cmd-with-flag", "flagecho"])

ohcegalf


SystemExit: 0

In [17]:
#calling the fn with a flag that accepts a value. Note that flags are case-insensitive
libcli.run(["--flag", "addtext=bla", "example-cmd-with-flag", "flagecho"])

flagecho
bla


SystemExit: 0

In [18]:
#calling the fn with unregistered arbitrary flags (and also multiple)
libcli.run([ "--flag", "dummy1","--flag", "dummy2=5", "--flag", "testflag", "example-cmd-with-flag", "flagecho"])

ohcegalf
bla
{'dummy1': True, 'dummy2': '5'}


SystemExit: 0

__synopsis, further reading__

Flags are different than commandline options in some ways:
- commandline options are part of the function that is exposed to the cli. you have to route them to subfunctions if needed there. Flags can be accessed from everywhere without routing.
- A single flag can be consumed by multiple functions. Eg. a "--flag verbose" flag can be accessed in multiple places and modify the programs output.
- Flags are an easy way to add debug switches. in that case, you dont even have to register a flag in the cli. just pass it and query the unregistered flags in the code.